In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.regularizers import l2
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
housing = fetch_california_housing()
X, y = housing.data, housing.target

threshold = np.percentile(y, 70)
yBinary = (y > threshold).astype(int)
print(f"the threshold: {threshold:.4f}")

# intentionally small train set to stress test the optimizers
xTrain, xTest, yTrain, yTest = train_test_split(
    X, yBinary, test_size=0.95, random_state=42
)
print(f"train: {xTrain.shape[0]}  vs  test: {xTest.shape[0]}")

# scale using only train stats so there's no data leakage
scaler = StandardScaler()
xTrain = scaler.fit_transform(xTrain)
xTest  = scaler.transform(xTest)

In [3]:
def buildSgdModel(inputDim):
    # plain SGD, no batch norm, no scheduler
    model = models.Sequential([
        layers.Input(shape=(inputDim,)),

        layers.Dense(256, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(128, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(64, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(32, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(16, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


def buildAdvancedModel(inputDim):
    # Adam + batch norm after each dense layer + LR scheduler
    model = models.Sequential([
        layers.Input(shape=(inputDim,)),

        layers.Dense(256, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(128, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(64, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(32, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(16, kernel_initializer='he_normal', kernel_regularizer=l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [4]:
inputDim = xTrain.shape[1]

earlyStopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print("--- Strategy 1: SGD ---")
modelSgd = buildSgdModel(inputDim)
historySgd = modelSgd.fit(
    xTrain, yTrain,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[earlyStopping],
    verbose=1
)

In [5]:
earlyStopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

# cut the LR in half if val_loss stops improving for 5 epochs
lrScheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5
)

modelAdvanced = buildAdvancedModel(inputDim)
historyAdvanced = modelAdvanced.fit(
    xTrain, yTrain,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[earlyStopping, lrScheduler],
    verbose=1
)

In [ ]:
# pull the numbers we care about from each training run
sgdEpochs = len(historySgd.history['loss'])
sgdLoss = historySgd.history['loss'][-1]
sgdTrainAcc = historySgd.history['accuracy'][-1]
_, sgdTestAcc = modelSgd.evaluate(xTest, yTest, verbose=0)
sgdGap = sgdTrainAcc - sgdTestAcc

advEpochs = len(historyAdvanced.history['loss'])
advLoss = historyAdvanced.history['loss'][-1]
advTrainAcc = historyAdvanced.history['accuracy'][-1]
_, advTestAcc = modelAdvanced.evaluate(xTest, yTest, verbose=0)
advGap = advTrainAcc - advTestAcc

results = {
    'Metric': ['Epochs to Converge', 'Final Train Loss', 'Final Test Accuracy', 'Generalization Gap'],
    '1. Standard (SGD)': [
        sgdEpochs,
        f"{sgdLoss:.4f}",
        f"{sgdTestAcc * 100:.2f}%",
        f"{sgdGap:.4f}"
    ],
    '2. Advanced (Adam+BN+Sched)': [
        advEpochs,
        f"{advLoss:.4f}",
        f"{advTestAcc * 100:.2f}%",
        f"{advGap:.4f}"
    ]
}

df = pd.DataFrame(results).set_index('Metric')
print(df.to_string())